# 02 - 视觉问答智能体

本教程介绍如何使用ReAct框架构建视觉问答智能体。

## 学习目标
- 理解ReAct推理框架
- 掌握智能体动作类型
- 学会构建多步推理流程

## 1. 环境准备

In [ ]:
import sys
sys.path.append('../src')

import numpy as np
from vision_qa_agent import VisionQAAgent, ActionType, VisionAction

## 2. ReAct框架介绍

ReAct = Reasoning + Acting，交替进行推理和行动：

```
循环:
  1. Thought: 分析当前状态
  2. Action: 执行动作
  3. Observation: 获取结果
  4. 判断是否完成
```

In [ ]:
# 查看可用的动作类型
print("可用动作类型:")
for action in ActionType:
    print(f"  - {action.value}")

## 3. 创建智能体

In [ ]:
# 创建基础智能体
agent = VisionQAAgent(max_steps=5)
print(f"最大推理步数: {agent.max_steps}")

In [ ]:
# 简单问答
result = agent.answer("什么是深度学习?")

print(f"问题: 什么是深度学习?")
print(f"答案: {result.answer}")
print(f"推理步数: {result.num_steps}")
print(f"成功: {result.success}")

## 4. 查看推理过程

In [ ]:
# 详细查看每一步
result = agent.answer("比较猫和狗的特点")

print("推理过程:")
print("=" * 50)
for step in result.steps:
    print(f"\nStep {step.step_num}:")
    print(f"  思考: {step.thought[:50]}..." if len(step.thought) > 50 else f"  思考: {step.thought}")
    print(f"  动作: {step.action.action_type.value}")
    if step.action.params:
        print(f"  参数: {step.action.params}")
    print(f"  观察: {step.observation[:50]}..." if len(step.observation) > 50 else f"  观察: {step.observation}")

## 5. 带图像的问答

In [ ]:
# 创建模拟图像
image = np.random.rand(128, 128, 3).astype(np.float32)
print(f"图像形状: {image.shape}")

# 带图像的问答
result = agent.answer("描述这张图片的内容", image=image)

print(f"\n答案: {result.answer}")

In [ ]:
# 图像定位问题
result = agent.answer("图片中的主要物体在什么位置?", image=image)
print(f"答案: {result.answer}")

## 6. 结合检索的问答

In [ ]:
from multimodal_retriever import MultimodalRetriever, MultimodalDocument

# 创建知识库
retriever = MultimodalRetriever()
knowledge = [
    "猫是一种独立性强的宠物，喜欢独处",
    "狗是忠诚的动物，被称为人类最好的朋友",
    "金鱼是常见的观赏鱼类",
    "仓鼠是小型啮齿类宠物",
]
for text in knowledge:
    retriever.add_document(MultimodalDocument(content=text))

print(f"知识库文档数: {retriever.num_documents}")

In [ ]:
# 创建带检索的智能体
agent_with_retriever = VisionQAAgent(retriever=retriever, max_steps=5)

result = agent_with_retriever.answer("哪种宠物最忠诚?")
print(f"答案: {result.answer}")
print(f"参考来源数: {len(result.sources)}")

## 7. 自定义动作处理

In [ ]:
# 手动创建动作
search_action = VisionAction(
    action_type=ActionType.SEARCH,
    params={"query": "宠物特点", "top_k": 2}
)
print(f"动作: {search_action}")

# 执行动作
observation = agent_with_retriever._execute_action(search_action, image=None)
print(f"观察结果: {observation}")

## 8. 练习

1. 创建一个包含5个知识点的知识库
2. 使用智能体回答相关问题
3. 观察推理过程，理解ReAct框架

In [ ]:
# 练习空间


## 总结

本教程介绍了:
- ReAct框架: 推理+行动的交替循环
- VisionQAAgent: 视觉问答智能体
- 动作类型: search, describe, compare, locate, count, answer
- 结合检索增强问答能力

下一步: 学习代码助手系统